## Rag Pipeline end to end workflow 

## Import Libraries and Load API Keys

This section prepares the environment for building a **Retrieval-Augmented Generation (RAG)** workflow using LangChain.

### Libraries Imported

**Document Loading**
- `DirectoryLoader`
- `TextLoader`

These help load text files from a directory into LangChain document objects.

**Text Splitting**
- `RecursiveCharacterTextSplitter`

Large documents are split into smaller chunks so that they can be processed efficiently by embedding models.

**Embeddings**
- `HuggingFaceEmbeddings`
- `OpenAIEmbeddings`

Embeddings convert text into vector representations that can be stored in a vector database.

**LLM Integration**
- `ChatOpenAI`

Used to send prompts and queries to OpenAI chat models.

**Vector Database**
- `Chroma`

Stores document embeddings and enables semantic search during retrieval.

**LangChain Core Components**
- `SystemMessage` and `HumanMessage` represent chat messages.
- `Document` represents structured text data.
- `ChatPromptTemplate` helps build structured prompts.

### Loading API Keys

The `.env` file is loaded using `load_dotenv()` so that API keys can be accessed securely.

Two keys are loaded:
- `OPENAI_API_KEY`
- `HF_TOKEN` (Hugging Face)

To confirm that the keys are loaded correctly, the script prints the **first and last four characters** of each key.

In [ ]:
# Import standard Python libraries
import os
from dotenv import load_dotenv

# Import document loaders to read files from a directory
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Import text splitter to break large documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import embedding models
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Import vector database
from langchain_chroma import Chroma

# Import message formats used by LLM chat models
from langchain_core.messages import SystemMessage, HumanMessage

# Import document structure used by LangChain
from langchain_core.documents import Document

# Import prompt template utility
from langchain_core.prompts import ChatPromptTemplate


# Load environment variables from .env file
load_dotenv(".env", override=True)

# Read OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

# Read Hugging Face API key from environment
HUGGING_FACE_API_KEY = os.getenv("HF_TOKEN", "").strip()

# Print partial API keys to verify they loaded correctly
print(f"OPENAI_API_KEY: {OPENAI_API_KEY[:4]}...{OPENAI_API_KEY[-4:]}") 
print(f"HUGGING_FACE_API_KEY: {HUGGING_FACE_API_KEY[:4]}...{HUGGING_FACE_API_KEY[-4:]}")

## Load Documents from the Local Knowledge Base

In this step, we load the source documents that will later be used for building a **Retrieval-Augmented Generation (RAG)** system.

### DirectoryLoader

`DirectoryLoader` is used to read files from a directory and convert them into **LangChain Document objects**.

Configuration used here:
- `docs` → the folder containing the source documents
- `glob="**/*.md"` → recursively loads all markdown files
- `TextLoader` → reads each file as plain text

### Document Structure

Each loaded document becomes a `Document` object containing:
- `page_content` → the text content
- `metadata` → information such as the file path

### Output

The code prints:
1. The total number of documents loaded
2. The metadata for each document

This helps verify that the correct files were loaded before moving on to **text chunking and embedding generation**.

In [ ]:
# Load documents from the "docs" directory using LangChain's DirectoryLoader
# It will recursively search for markdown files (.md)

loader = DirectoryLoader(
    "docs",                 # Folder containing documentation files
    glob="**/*.md",         # Pattern to match all markdown files recursively
    loader_cls=TextLoader   # Loader used to read text files
)

# Load all matched documents into memory
documents = loader.load()

# Print the number of documents loaded
print(f"Loaded {len(documents)} documents")

# Display metadata for each loaded document
# Metadata usually includes the file path and other file-related information
for doc in documents:
    print(doc.metadata)

## Split Documents into Chunks

Large documents are difficult for language models to process directly.  
To make them suitable for embedding and retrieval, we split them into smaller text segments.

### RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` is used to divide documents into manageable chunks while preserving context.

Configuration used:
- **chunk_size = 500**  
  Each chunk contains up to 500 characters.

- **chunk_overlap = 50**  
  Adjacent chunks share 50 characters to maintain context continuity.

This overlap helps prevent loss of meaning when important information spans chunk boundaries.

### Output

The code:
1. Splits the documents into chunks.
2. Prints the total number of chunks created.
3. Displays a sample chunk to verify the splitting process.

These chunks will later be converted into **vector embeddings** and stored in a **vector database** for retrieval during question answering.

In [ ]:
# Split the loaded documents into smaller chunks
# This helps LLMs process the content efficiently and improves retrieval accuracy

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Maximum number of characters in each chunk
    chunk_overlap=50    # Overlap between chunks to preserve context across boundaries
)

# Apply the splitter to the loaded documents
chunks = text_splitter.split_documents(documents)

# Print how many chunks were created
print(f"Created {len(chunks)} chunks")

# Display a sample chunk to verify the result
print("\nSample chunk:\n")
print(chunks[0].page_content)

## Add Topic Metadata to Document Chunks

After splitting documents into chunks, we enrich each chunk with additional metadata.

### Why Metadata is Important

Metadata helps organize document chunks and enables more advanced retrieval strategies, such as:
- filtering by topic
- improving search relevance
- enabling structured retrieval

### Topic Classification

Each chunk inherits metadata from its original document, including the `source` file path.  
The code analyzes this source and assigns a **topic label** based on keywords:

| Keyword in Source | Assigned Topic |
|-------------------|---------------|
| refund            | refunds       |
| shipping          | shipping      |
| security          | security      |
| faq               | faq           |
| otherwise         | general       |

### Verification

Finally, the metadata of the first chunk is printed to verify that the topic labeling has been applied correctly.

This metadata will later help improve the **retrieval step of the RAG pipeline**.

In [ ]:
# Add topic metadata to each document chunk
# This helps categorize chunks and can improve filtering during retrieval

for chunk in chunks:
    # Extract the source filename from metadata
    source = chunk.metadata.get("source", "").lower()

    # Assign a topic label based on keywords found in the source path
    if "refund" in source:
        chunk.metadata["topic"] = "refunds"
    elif "shipping" in source:
        chunk.metadata["topic"] = "shipping"
    elif "security" in source:
        chunk.metadata["topic"] = "security"
    elif "faq" in source:
        chunk.metadata["topic"] = "faq"
    else:
        chunk.metadata["topic"] = "general"

# Display metadata of the first chunk to verify the topic assignment
chunks[0].metadata

## Generate Text Embeddings with HuggingFace

In this step, we initialize an **embedding model** that converts text into vector representations.

### What Are Embeddings?

Embeddings transform text into numerical vectors that capture semantic meaning.  
These vectors allow us to compare documents based on similarity rather than exact keyword matches.

### Model Used

The embedding model used is:

**`sentence-transformers/all-MiniLM-l6-v2`**

This model is:
- lightweight and fast
- widely used for semantic search
- well suited for building vector databases in RAG systems

### Sanity Check

To confirm that the embedding model is working correctly, we generate an embedding for the query:


In [ ]:
# Create an embedding model using HuggingFaceEmbeddings
# This model converts text into numerical vectors that capture semantic meaning

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2"   # Lightweight and efficient sentence embedding model
)

# Perform a quick sanity check by generating an embedding for a sample query
test_vec = embeddings.embed_query("How long do refunds take?")

# Print the size of the embedding vector
print(f"Embedding length: {len(test_vec)}")

# Print the first few values of the embedding vector
print(test_vec[:5])

## Store Embeddings in Chroma Vector Database

After generating embeddings for document chunks, we store them in a **vector database**.

### What is ChromaDB?

ChromaDB is a lightweight vector database used for:
- storing embeddings
- performing semantic similarity search
- enabling Retrieval-Augmented Generation (RAG)

### Creating the Vector Store

The `Chroma.from_documents()` function:
1. Converts document chunks into embeddings
2. Stores those embeddings inside a Chroma collection
3. Saves the database locally so it can be reused later

Configuration used:
- **documents** → the text chunks created earlier
- **embedding** → the embedding model used for vector generation
- **collection_name** → name of the stored vector collection
- **persist_directory** → local directory where the database is saved

### Retriever Setup

A **retriever** is created using:

```python
vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
# Store the document chunks and their embeddings inside ChromaDB
# This creates a vector database that allows semantic search over the documents

vectorstore = Chroma.from_documents(
    documents=chunks,                     # Document chunks created earlier
    embedding=embeddings,                 # Embedding model used to convert text into vectors
    collection_name="customer_support_docs",  # Name of the vector collection
    persist_directory="./chroma_db"       # Local directory where the vector DB will be stored
)

# Create a retriever interface from the vector database
# This allows us to perform similarity search during RAG queries
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}   # Return the top 2 most relevant chunks for each query
)

# Confirm that the vector database was created successfully
print("Chroma DB setup complete.")

# Print the total number of stored document vectors
print(f"Number of documents in ChromaDB: {vectorstore._collection.count()}")

## Test the Retrieval System

Now that the vector database has been created, we test whether the retrieval system works correctly.

### Querying the Vector Database

A user question is defined:


The retriever searches the vector database and returns the **most relevant document chunks** based on semantic similarity.

### Retrieval Process

The code performs three steps:

1. Send the query to the retriever
2. Retrieve the **top 2 relevant document chunks**
3. Print the results

### Output

For each retrieved result, the code displays:
- the **topic metadata** assigned earlier
- the **content of the document chunk**

This confirms that the vector database can correctly find relevant information based on the user query.

This retrieval step is a key component of **Retrieval-Augmented Generation (RAG)** systems.

In [ ]:
# Test the retriever by asking a sample query
query = "How long do refunds take?"

# Use the retriever to find the most relevant document chunks
results = retriever.invoke(query)

# Print the retrieved results
for i, doc in enumerate(results, 1):

    # Display which result number we are viewing
    print(f"\n--- Result {i} ---")

    # Print the topic metadata that was assigned earlier
    print("Topic:", doc.metadata.get("topic"))

    # Print the actual content of the retrieved chunk
    print(doc.page_content)

## Topic Classification for Hierarchical Retrieval

In this step, we introduce a simple **topic classifier** that categorizes user questions before retrieving documents.

### Why Topic Classification?

In Retrieval-Augmented Generation (RAG) systems, filtering documents by topic can improve search quality by:
- reducing irrelevant retrieval results
- speeding up vector searches
- narrowing the search space

### How the Classifier Works

The `classify_topic()` function performs **keyword-based classification**:

1. The user question is converted to lowercase.
2. The function checks whether certain keywords appear in the text.
3. Based on the keywords found, the question is assigned a topic.

### Supported Topics

| Keywords | Topic |
|--------|--------|
| refund, money back, reimbursement | refunds |
| shipping, delivery, tracking | shipping |
| login, password, suspicious | security |
| faq, contact, support | faq |
| none of the above | general |

### Purpose

This topic label can later be used to **filter the vector database** so the retriever only searches documents from the relevant topic.

This approach creates a **hierarchical retrieval system**:
1. classify question → determine topic
2. retrieve documents within that topic
3. pass relevant context to the LLM

In [ ]:
# Define a simple topic classifier to categorize user questions
# This helps the retrieval system focus on the most relevant documents

def classify_topic(question: str) -> str:

    # Convert the question to lowercase to make keyword matching case-insensitive
    q = question.lower()

    # Check if the question relates to refunds
    if any(word in q for word in ["refund", "refunds", "money back", "reimbursement"]):
        return "refunds"

    # Check if the question relates to shipping or delivery
    elif any(word in q for word in ["shipping", "delivery", "dispatch", "tracking", "address"]):
        return "shipping"

    # Check if the question relates to account security
    elif any(word in q for word in ["security", "login", "password", "locked", "suspicious"]):
        return "security"

    # Check if the question relates to general FAQ or support questions
    elif any(word in q for word in ["contact", "faq", "question", "support"]):
        return "faq"

    # Default category if no keywords match
    else:
        return "general"

## Metadata-Filtered Retrieval, Query Rewriting, and Query Expansion

This section improves the retrieval stage of the RAG pipeline by adding three useful techniques:

1. **metadata-filtered retrieval**
2. **query rewriting**
3. **query expansion**

### 1. Metadata-Filtered Retrieval

The function `retrieve_with_topic()` first classifies the user question into a topic using `classify_topic()`.

It then performs a similarity search in the vector database with a metadata filter:

- semantic search finds relevant chunks
- metadata filtering restricts results to the predicted topic

This makes retrieval more focused and can improve relevance.

Example:
- Question: **How long do refunds take?**
- Detected topic: **refunds**

The retrieved documents are then printed to verify the results.

### 2. Query Rewriting

Sometimes users ask vague, misspelled, or grammatically incorrect questions.  
To improve retrieval quality, the code uses an LLM to rewrite the question into a clearer search query.

The rewriting prompt instructs the model to:
- preserve the original meaning
- fix spelling and grammar
- return only the rewritten query

Example:
- Original: `when are refsds processed`
- Rewritten: a cleaner version suitable for retrieval

This can improve search performance when the original user query is poorly written.

### 3. Query Expansion

A single question can often be expressed in multiple ways.  
To make retrieval more robust, the code generates **3 alternative search queries** for the same user question.

The `expand_query()` function:
- runs the expansion prompt through the LLM
- splits the output into separate lines
- removes duplicates
- returns up to 3 unique rewritten queries

This helps support multi-query retrieval strategies, where the system searches using several semantically related phrasings instead of only one query.

### Why These Steps Matter

These enhancements improve retrieval quality by:

- narrowing the search space using metadata
- correcting unclear user questions
- increasing recall with alternative query formulations

Together, they make the RAG pipeline more accurate and more resilient to imperfect user input.

In [ ]:
# Retrieve documents using topic-based metadata filtering
def retrieve_with_topic(question: str, k: int = 4):

    # First classify the question into a topic
    topic = classify_topic(question)

    # Search the vector database using both semantic similarity and topic filter
    docs = vectorstore.similarity_search(
        question,
        k=k,
        filter={"topic": topic}
    )

    # Return both the detected topic and the retrieved documents
    return topic, docs


# Test the metadata-filtered retrieval
topic, docs = retrieve_with_topic("How long do refunds take?")
print("Topic:", topic)

# Print the retrieved documents
for i, d in enumerate(docs, 1):
    print(f"\n--- Doc {i} ---")
    print(d.page_content)


# Create an LLM client that will be used for query rewriting and expansion
client = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


# Import utilities for prompt templates and string output parsing
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Prompt template for rewriting unclear user questions into better search queries
rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite the user's question into a clearer search query for retrieving relevant support documents.
Keep the meaning the same.
Fix grammar and spelling.
Return only the rewritten query.

Question: {question}
""")


# Build a simple chain:
# prompt -> LLM -> plain string output
rewrite_chain = rewrite_prompt | client | StrOutputParser()


# Test the rewriting chain with a misspelled question
test_question = "when are refsds processed"
rewritten = rewrite_chain.invoke({"question": test_question})

print("Original:", test_question)
print("Rewritten:", rewritten)


# Prompt template for generating multiple alternative search queries
expand_prompt = ChatPromptTemplate.from_template("""
Generate 3 alternative search queries for the user question below.
They should preserve meaning but use different wording.
Return each query on a new line only.

Question: {question}
""")


# Build the expansion chain
expand_chain = expand_prompt | client | StrOutputParser()


# Expand a user question into multiple semantically similar search queries
def expand_query(question: str) -> list[str]:

    # Run the expansion chain
    raw = expand_chain.invoke({"question": question})

    # Split the output into lines and clean list markers or extra spaces
    lines = [line.strip("-• ").strip() for line in raw.split("\n") if line.strip()]

    # Remove duplicates while keeping the original order
    unique = []
    seen = set()
    for line in lines:
        if line not in seen:
            seen.add(line)
            unique.append(line)

    # Return at most 3 unique query variants
    return unique[:3]


# Test query expansion
expand_query("How long do refunds take?")

## Multi-Query Retrieval, Deduplication, and LLM-Based Reranking

This section combines several retrieval improvements into a more advanced RAG pipeline.

### 1. Expanded Retrieval Pipeline

The function `retrieve_expanded()` performs multiple steps:

1. **Rewrite the user question**  
   The original question is rewritten into a clearer search query.

2. **Expand the rewritten query**  
   Multiple alternative phrasings are generated.

3. **Retrieve documents for each query**  
   Each query is sent through topic-based retrieval.

4. **Deduplicate results**  
   Duplicate chunks are removed so the same text does not appear multiple times.

The function returns:
- the rewritten query
- the expanded queries
- the final set of unique retrieved documents

This improves retrieval by increasing coverage while avoiding redundant results.

### 2. LLM-Based Document Scoring

Not all retrieved chunks are equally useful.  
To improve relevance, the code uses an LLM to assign a **relevance score from 1 to 10** for each document chunk.

The `score_prompt` asks the model to compare:
- the user question
- the retrieved chunk

and return only a numeric relevance score.

The function `score_doc()` runs this scoring step for one chunk and safely converts the result into a float.

### 3. Reranking Documents

The function `rerank_documents()`:
- scores each retrieved document
- sorts documents by score in descending order
- keeps only the top `N` most relevant chunks

This step improves context quality before passing retrieved information to the answer-generation model.

### 4. Formatting Context

The function `format_context()` combines the selected top-ranked chunks into one text block.

This formatted context can then be inserted into a final prompt for the language model.

### Why This Matters

This pipeline improves a standard RAG system in several ways:

- **query rewriting** improves poorly worded questions
- **query expansion** increases recall
- **deduplication** removes repeated chunks
- **LLM reranking** improves final context quality

Together, these steps make retrieval more accurate and produce stronger context for answer generation.

In [ ]:
# Combine query rewriting, query expansion, topic-based retrieval,
# and document deduplication into one retrieval pipeline
def retrieve_expanded(question: str, k_per_query: int = 3):

    # First rewrite the original question into a cleaner search query
    rewritten = rewrite_chain.invoke({"question": question})

    # Generate multiple alternative versions of the rewritten query
    expanded = expand_query(rewritten)

    # Combine the rewritten query with the expanded queries
    all_queries = [rewritten] + expanded

    # Store all retrieved documents and track duplicates
    all_docs = []
    seen_text = set()

    # Run retrieval for each query variant
    for q in all_queries:
        topic, docs = retrieve_with_topic(q, k=k_per_query)

        # Deduplicate documents based on text content
        for doc in docs:
            key = doc.page_content.strip()
            if key not in seen_text:
                seen_text.add(key)
                all_docs.append(doc)

    # Return the rewritten query, query expansions, and unique retrieved documents
    return {
        "rewritten_query": rewritten,
        "expanded_queries": expanded,
        "documents": all_docs
    }


# Prompt used to ask the LLM to score document relevance
score_prompt = ChatPromptTemplate.from_template("""
Given the user question and a retrieved document chunk, score how relevant the chunk is from 1 to 10.

Return only the number.

Question: {question}

Chunk:
{chunk}
""")


# Build a scoring chain:
# prompt -> LLM -> plain string output
score_chain = score_prompt | client | StrOutputParser()


# Score one document chunk against a user question
def score_doc(question: str, chunk: str) -> float:

    # Ask the LLM to assign a relevance score
    raw = score_chain.invoke({"question": question, "chunk": chunk}).strip()

    # Convert the result into a float if possible
    try:
        return float(raw)
    except:
        return 0.0


# Rerank retrieved documents and keep only the top-scoring ones
def rerank_documents(question: str, docs: list, top_n: int = 3):

    scored = []

    # Score each document individually
    for doc in docs:
        score = score_doc(question, doc.page_content)
        scored.append((score, doc))

    # Sort documents by descending score
    scored.sort(key=lambda x: x[0], reverse=True)

    # Return only the top N documents
    return scored[:top_n]


# Convert reranked documents into one combined context string
def format_context(reranked_docs):
    return "\n\n".join([doc.page_content for score, doc in reranked_docs])

## Final Answer Generation and Retrieval Routing

This section completes the RAG pipeline by generating the final answer from the retrieved context.

### 1. Final Answer Prompt

The `answer_prompt` defines how the language model should respond.

The model is instructed to:
- act as a helpful support assistant
- answer the question **using only the retrieved context**
- say **"I don't know."** if the answer is not present in the context

This is an important guardrail in RAG systems because it reduces hallucination and keeps answers grounded in retrieved documents.

### 2. Answer Generation Chain

The `answer_chain` connects:
- the prompt template
- the chat model
- the output parser

This creates a reusable pipeline for generating final grounded answers.

### 3. End-to-End Question Answering

The function `answer_question()` runs the full retrieval and answer generation workflow:

1. retrieve documents using:
   - query rewriting
   - query expansion
   - topic filtering
   - deduplication

2. rerank the retrieved documents using LLM scoring

3. combine the best chunks into one context block

4. generate the final answer using the context and the original question

The function returns:
- the rewritten query
- expanded queries
- reranked documents
- final context
- generated answer

This makes the pipeline transparent and easier to debug.

### 4. Testing the Pipeline

The code tests the complete workflow using the question:

`How long do refunds take?`

and prints the final answer returned by the model.

### 5. Retrieval Router

The `router()` function defines a simple strategy for deciding which retrieval features to use.

Current routing rules:
- always rewrite the question
- expand the query only if it is short
- always apply topic filtering
- always rerank results

This is a lightweight example of **retrieval orchestration**, where the system decides how much processing is needed based on the input query.

### Why This Step Matters

This section turns the earlier components into a complete **RAG application pipeline**:

- retrieve relevant content
- improve relevance with reranking
- ground the model in retrieved context
- generate a final answer safely

It is the final step that connects document retrieval with response generation.

In [ ]:
# Create the final prompt used for answer generation
# The model is instructed to answer only from the retrieved context
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful support assistant.
Answer the user's question using only the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}
""")

# Build the final answer generation chain:
# prompt -> LLM -> plain string output
answer_chain = answer_prompt | client | StrOutputParser()


# End-to-end function that retrieves documents, reranks them,
# builds context, and generates the final answer
def answer_question(question: str):

    # Retrieve documents using rewriting, expansion, filtering, and deduplication
    retrieved = retrieve_expanded(question)

    # Rerank the retrieved documents and keep the top 3
    reranked = rerank_documents(question, retrieved["documents"], top_n=3)

    # Combine the top-ranked documents into one context block
    context = format_context(reranked)

    # Generate the final answer using the context and the original question
    answer = answer_chain.invoke({"context": context, "question": question})

    # Return all intermediate and final outputs for inspection
    return {
        "rewritten_query": retrieved["rewritten_query"],
        "expanded_queries": retrieved["expanded_queries"],
        "reranked_docs": reranked,
        "context": context,
        "answer": answer
    }


# Test the full RAG pipeline with a sample question
result = answer_question("How long do refunds take?")
print(result["answer"])


# Define a simple router to decide which retrieval steps to apply
def router(question: str):
    q = question.lower()

    return {
        "use_rewrite": True,                               # Always rewrite the question
        "use_expansion": True if len(q.split()) < 8 else False,  # Expand only for shorter queries
        "use_topic_filter": True,                         # Always apply topic filtering
        "use_rerank": True                                # Always rerank retrieved documents
    }

## Agentic RAG Pipeline

This section defines an **agentic retrieval pipeline** that dynamically chooses which retrieval steps to use based on the input question.

### What Makes It Agentic?

Unlike a fixed retrieval pipeline, this function does not always follow the exact same steps.  
Instead, it first calls the `router()` function, which decides whether to use:

- query rewriting
- query expansion
- topic-based filtering
- reranking

This makes the system more flexible and adaptive.

### Step-by-Step Flow

### 1. Routing Decisions
The pipeline begins by calling:

```python
decisions = router(question)

In [ ]:
# Agentic RAG pipeline
# This function dynamically decides which retrieval steps to apply
# based on the router output
def agentic_rag(question: str):

    # Get routing decisions for the current question
    decisions = router(question)

    # Start with the original question
    working_query = question
    expanded_queries = [question]

    # Rewrite the question if enabled by the router
    if decisions["use_rewrite"]:
        working_query = rewrite_chain.invoke({"question": question})

    # Expand the query into multiple variants if enabled
    if decisions["use_expansion"]:
        expanded_queries = [working_query] + expand_query(working_query)
    else:
        expanded_queries = [working_query]

    # Collect retrieved documents and remove duplicates
    all_docs = []
    seen = set()

    # Run retrieval for each query variant
    for q in expanded_queries:

        # Use topic-filtered retrieval if enabled
        if decisions["use_topic_filter"]:
            _, docs = retrieve_with_topic(q, k=3)
        else:
            docs = retriever.invoke(q)

        # Deduplicate documents using their text content
        for d in docs:
            key = d.page_content.strip()
            if key not in seen:
                seen.add(key)
                all_docs.append(d)

    # Rerank the documents if enabled
    if decisions["use_rerank"]:
        reranked = rerank_documents(question, all_docs, top_n=3)
    else:
        reranked = [(0, d) for d in all_docs[:3]]

    # Build the final context from the selected documents
    context = format_context(reranked)

    # Generate the final answer using the answer generation chain
    answer = answer_chain.invoke({"context": context, "question": question})

    # Return both the decisions and outputs for inspection
    return {
        "decisions": decisions,
        "rewritten_query": working_query,
        "expanded_queries": expanded_queries,
        "context": context,
        "answer": answer,
        "sources": reranked
    }

## Gradio Interface for the RAG Assistant

This section creates a **simple web interface** that allows users to interact with the RAG pipeline through a browser.

The interface is built using **Gradio**, which is a lightweight Python library for creating ML and AI demos.

### 1. Gradio Wrapper Function

The function `gradio_rag()` acts as a bridge between the UI and the RAG system.

It performs the following steps:

1. calls the `agentic_rag()` pipeline with the user's question
2. formats the retrieved source documents for readability
3. returns the results that will be displayed in the UI

The function returns five values:

- the final generated answer
- the rewritten search query
- the expanded queries
- the retrieved document chunks
- the router decisions used by the pipeline

### 2. Creating the Gradio Interface

The `gr.Interface()` object defines the user interface.

#### Inputs
The UI contains a single input field:

- **Textbox** — where the user enters a support question

#### Outputs
The interface displays five outputs:

1. **Answer** — the final response generated by the RAG system
2. **Rewritten Query** — the improved search query generated by the rewrite step
3. **Expanded Queries** — alternative search queries generated during query expansion
4. **Retrieved Sources** — the document chunks retrieved from the vector database
5. **Agent Decisions** — which retrieval steps the router decided to apply

### 3. Application Metadata

The interface also includes:

- **title** — the name of the application
- **description** — a summary of the technologies used

This project integrates multiple components:

- LangChain for orchestration
- HuggingFace embeddings for semantic search
- ChromaDB for vector storage
- OpenAI models for reasoning and answering
- Gradio for the user interface

### 4. Launching the Application

The command


In [ ]:
# Import Gradio for building a simple web UI
import gradio as gr


# Wrapper function that connects the RAG pipeline to the Gradio interface
def gradio_rag(question):

    # Run the agentic RAG pipeline
    result = agentic_rag(question)

    # Prepare readable source output for the UI
    sources_text = []

    # Loop through reranked sources and format them nicely
    for i, (score, doc) in enumerate(result["sources"], 1):

        sources_text.append(
            f"Source {i} | score={score}\n"
            f"topic={doc.metadata.get('topic')}\n"
            f"{doc.page_content}"
        )

    # Return values that will be displayed in the UI
    return (
        result["answer"],                       # Final generated answer
        result["rewritten_query"],              # Rewritten search query
        "\n".join(result["expanded_queries"]),  # Expanded queries
        "\n\n---\n\n".join(sources_text),       # Retrieved document chunks
        str(result["decisions"])                # Router decisions
    )


# Create the Gradio interface
demo = gr.Interface(

    # Function executed when user submits a question
    fn=gradio_rag,

    # User input component
    inputs=gr.Textbox(label="Ask a support question", lines=2),

    # Output components displayed to the user
    outputs=[
        gr.Textbox(label="Answer"),              # Final RAG answer
        gr.Textbox(label="Rewritten Query"),     # Query after rewriting
        gr.Textbox(label="Expanded Queries"),    # Query expansion results
        gr.Textbox(label="Retrieved Sources"),   # Retrieved documents
        gr.Textbox(label="Agent Decisions")      # Routing decisions
    ],

    # UI title
    title="Support Knowledge RAG Assistant",

    # Short description of the technology stack used
    description="LangChain + Hugging Face embeddings + Chroma + ChatOpenAI + Gradio"
)


# Launch the web interface
demo.launch()